# ДЗ 3. Парсинг, Git и тестирование на Python

Сайт: http://books.toscrape.com

## Задание 1

`get_book_data(book_url)` собирает название, цену, рейтинг, количество в наличии, описание и все поля таблицы Product Information.

Пример: http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html

Код функций совпадает с `scraper.py`.


In [1]:
"""Парсер каталога books.toscrape.com."""

import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

CATALOGUE_URL = "http://books.toscrape.com/catalogue/page-{N}.html"
DATA_PATH = Path("artifacts") / "books_data.txt"
REQUEST_TIMEOUT_SECONDS = 30
_SESSION = requests.Session()


def _fetch_response(url: str) -> requests.Response:
    """Загружает страницу и кодировку.

    Args:
        url (str): абсолютный адрес страницы.

    Returns:
        requests.Response: ответ сервера.
        Текст читается в кодировке страницы.

    Raises:
        requests.RequestException: страница не загрузилась
        после трёх попыток.
    """
    last_error: requests.RequestException | None = None
    for attempt in range(3):
        try:
            response = _SESSION.get(
                url,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.encoding = response.apparent_encoding or "utf-8"
            return response
        except requests.RequestException as error:
            last_error = error
            time.sleep(1 + attempt)
    if last_error is None:
        raise requests.RequestException(
            f"Не удалось загрузить {url}"
        )
    raise last_error


def _parse_stock_quantity(availability_text: str) -> int:
    """Достаёт число книг из строки наличия.

    Args:
        availability_text (str): текст вроде
        «In stock (22 available)».

    Returns:
        int: количество доступных экземпляров.
        0, если числа нет.
    """
    match = re.search(r"(\d+)", availability_text)
    if match is None:
        return 0
    return int(match.group(1))


def get_book_data(book_url: str) -> dict:
    """Собирает данные одной книги.

    Args:
        book_url (str): адрес страницы книги.

    Returns:
        dict: название, цена, рейтинг, количество
        в наличии, описание и все поля таблицы
        Product Information.
    """
    response = _fetch_response(book_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.find("h1").get_text(strip=True)
    price = soup.select_one("p.price_color").get_text(strip=True)
    rating_class = soup.select_one("p.star-rating").get("class", [])
    rating = next(
        (name for name in rating_class if name != "star-rating"),
        "",
    )
    availability_text = soup.select_one(
        "p.availability"
    ).get_text(strip=True)
    description = ""
    description_header = soup.find(id="product_description")
    if description_header is not None:
        paragraph = description_header.find_next_sibling("p")
        if paragraph is not None:
            description = paragraph.get_text(strip=True)

    product_information: dict[str, str] = {}
    table = soup.find("table", class_="table")
    if table is not None:
        for row in table.find_all("tr"):
            header = row.find("th")
            value = row.find("td")
            if header is None or value is None:
                continue
            product_information[
                header.get_text(strip=True)
            ] = value.get_text(strip=True)

    return {
        "title": title,
        "price": price,
        "rating": rating,
        "stock_quantity": _parse_stock_quantity(availability_text),
        "description": description,
        "product_information": product_information,
    }


def _page_url(catalogue_url: str, page_number: int) -> str:
    """Подставляет номер страницы в шаблон.

    Args:
        catalogue_url (str): шаблон с маркером {N}.
        page_number (int): номер страницы, начиная с 1.

    Returns:
        str: адрес конкретной страницы каталога.
    """
    return catalogue_url.replace("{N}", str(page_number))


def _book_urls(page_url: str, page_html: str) -> list[str]:
    """Собирает абсолютные ссылки на книги.

    Args:
        page_url (str): адрес страницы каталога.
        База для относительных ссылок.
        page_html (str): HTML этой страницы.

    Returns:
        list[str]: ссылки на страницы книг
        в порядке показа на странице.
    """
    soup = BeautifulSoup(page_html, "html.parser")
    urls: list[str] = []
    for link in soup.select("article.product_pod h3 a"):
        href = link.get("href")
        if not href:
            continue
        urls.append(urljoin(page_url, href))
    return urls


def scrape_books(
    catalogue_url: str = CATALOGUE_URL,
    is_save: bool = False,
    max_pages: int | None = None,
) -> list[dict]:
    """Обходит каталог и собирает данные книг.

    Args:
        catalogue_url (str): шаблон адреса страницы.
        Номер страницы ставится вместо {N}.
        is_save (bool): True — записать результат
        в artifacts/books_data.txt.
        False — файл не создавать и не изменять.
        max_pages (int | None): сколько страниц обойти.
        None — все страницы, пока сайт их отдаёт.

    Returns:
        list[dict]: данные книг в порядке обхода.
        Каждый элемент — результат get_book_data.
    """
    books: list[dict] = []
    page_number = 1
    while True:
        if max_pages is not None and page_number > max_pages:
            break
        page_url = _page_url(catalogue_url, page_number)
        response = _fetch_response(page_url)
        if response.status_code == 404:
            break
        response.raise_for_status()
        book_urls = _book_urls(page_url, response.text)
        if not book_urls:
            break
        for book_url in book_urls:
            books.append(get_book_data(book_url))
        print(
            f"Страница {page_number}: "
            f"собрано {len(book_urls)} книг"
        )
        page_number += 1

    if is_save:
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        DATA_PATH.write_text(
            json.dumps(books, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
    return books




In [2]:
import pprint

book_url = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
book = get_book_data(book_url)
pprint.pp(book, width=88)


{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 'Three',
 'stock_quantity': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This "
                'now-classic collection of poetry and drawings from Shel Silverstein '
                'celebrates its 20th anniversary with this special edition. '
                "Silverstein's humorous and creative verse can amuse the dowdiest of "
                'readers. Lemon-faced adults and fidgety kids sit still and read these '
                "rhythmic words and laugh and smile and love th It's hard to imagine a "
                'world without A Light in the Attic. This now-classic collection of '
                'poetry and drawings from Shel Silverstein celebrates its 20th '
                "anniversary with this special edition. Silverstein's humorous and "
                'creative verse can amuse the dowdiest of readers. Lemon-faced adults '
                'and fidgety kids sit still and re

## Задание 2

`scrape_books` обходит `http://books.toscrape.com/catalogue/page-{N}.html` и для каждой книги вызывает `get_book_data`.

`is_save=True` записывает результат в `artifacts/books_data.txt`. `is_save=False` файл не пишет.


In [3]:
res = scrape_books(is_save=True)
print(type(res), len(res))


Страница 1: собрано 20 книг
Страница 2: собрано 20 книг
Страница 3: собрано 20 книг
Страница 4: собрано 20 книг
Страница 5: собрано 20 книг
Страница 6: собрано 20 книг
Страница 7: собрано 20 книг
Страница 8: собрано 20 книг
Страница 9: собрано 20 книг
Страница 10: собрано 20 книг
Страница 11: собрано 20 книг
Страница 12: собрано 20 книг
Страница 13: собрано 20 книг
Страница 14: собрано 20 книг
Страница 15: собрано 20 книг
Страница 16: собрано 20 книг
Страница 17: собрано 20 книг
Страница 18: собрано 20 книг
Страница 19: собрано 20 книг
Страница 20: собрано 20 книг
Страница 21: собрано 20 книг
Страница 22: собрано 20 книг
Страница 23: собрано 20 книг
Страница 24: собрано 20 книг
Страница 25: собрано 20 книг
Страница 26: собрано 20 книг
Страница 27: собрано 20 книг
Страница 28: собрано 20 книг
Страница 29: собрано 20 книг
Страница 30: собрано 20 книг
Страница 31: собрано 20 книг
Страница 32: собрано 20 книг
Страница 33: собрано 20 книг
Страница 34: собрано 20 книг
Страница 35: собрано 20

## Задание 3

Ежедневный запуск в 19:00 лежит в `scraper.py` и работает только при запуске файла:

```python
if __name__ == "__main__":
    schedule.every().day.at("19:00").do(scrape_books, is_save=True)
    while True:
        schedule.run_pending()
        time.sleep(1)
```

В ноутбуке этот цикл не запускается. Ниже зарегистрирована задача на 19:00 и короткая проверка на ближайшее время. Ожидание ограничено, ячейка завершается.


In [4]:
import time
from datetime import datetime, timedelta

import schedule

schedule.clear()
daily = schedule.every().day.at("19:00").do(scrape_books, is_save=True)
print("Задача на 19:00:", daily)
print("Следующий запуск:", daily.next_run)

run_at = (datetime.now() + timedelta(seconds=3)).strftime("%H:%M:%S")
fired = {"done": False}


def short_job() -> None:
    """Отмечает срабатывание короткой проверки расписания.

    Returns:
        None: печатает время задачи и ставит флаг выполнения.
    """
    fired["done"] = True
    print("Короткая проверка выполнена, время задачи:", run_at)


short = schedule.every().day.at(run_at).do(short_job)
print("Короткая проверка зарегистрирована на", run_at)
print(short)

deadline = time.time() + 10
while time.time() < deadline and not fired["done"]:
    schedule.run_pending()
    time.sleep(0.5)

schedule.clear()
print("Ячейка завершена. Короткая задача сработала:", fired["done"])


Задача на 19:00: Job(interval=1, unit=days, do=scrape_books, args=(), kwargs={'is_save': True})
Следующий запуск: 2026-11-13 19:00:00
Короткая проверка зарегистрирована на 12:21:57
Job(interval=1, unit=days, do=short_job, args=(), kwargs={})
Короткая проверка выполнена, время задачи: 12:21:57
Ячейка завершена. Короткая задача сработала: True


## Задание 4

Тесты: `tests/test_scraper.py`.

В шаблоне опечатка: `pytest test/test_scraper.py`. Ячейка запускает `pytest tests/test_scraper.py`.


In [5]:
!pytest tests/test_scraper.py


============================= test session starts ==============================
platform darwin -- Python 3.14.6, pytest-9.2.0, pluggy-1.7.0
rootdir: /Users/dmitriygrosh/Documents/books-scraper
collected 3 items

tests/test_scraper.py ...                                                [100%]

============================== 3 passed in 4.44s ===============================


## Задание 5

```
books-scraper/
├── artifacts/books_data.txt
├── notebooks/GROSH_D_A_HW_03_python_ds.ipynb
├── scraper.py
├── README.md
├── tests/test_scraper.py
├── .gitignore
└── requirements.txt
```

## Рефлексия

Раздел заполняет только студент.

1. Что было самым сложным в задании?
Ответ: ...

2. Что получилось лучше всего?
Ответ: ...

3. Какие ошибки возникли и как вы их исправили?
Ответ: ...

4. Чему вы научились?
Ответ: ...

5. Что бы вы улучшили в решении?
Ответ: ...

6. Сколько времени заняло выполнение?
Ответ: ...
